# M-PESA prediction
<img src="Datasets/Images/MPesaFinance.png" alt="Mental Illness Image" width="500" height="450">
<h2> Case study </h2>

<i><h4>Introduction</i></h4>

* M-Pesa is a mobile money service launched by Safaricom, Kenya’s leading telecommunications company, in 2007. It has revolutionized financial transactions, allowing users to send, receive, deposit, and withdraw money using their mobile phones. M-Pesa has played a significant role in financial inclusion, particularly for unbanked populations.

<i><h4>How M-Pesa Works</h4></i>

* M-Pesa enables users to perform transactions via USSD codes or the M-Pesa app. Users register with Safaricom and link their mobile   numbers to an M-Pesa account. Key services include:

- Depositing money at M-Pesa agent shops.

- Sending money to other users and non-users.

- Withdrawing cash from agents or ATMs.

- Paying bills (electricity, water, internet, school fees, etc.).

- Merchant payments through Lipa na M-Pesa.

- Accessing micro-loans and savings via M-Shwari and KCB M-Pesa.

- Overdraft services using Fuliza.<br>

<i><h4>Challenges and Risks</i></h4>

* High transaction costs: Some users find M-Pesa charges expensive for frequent transactions.
* less finance monitoring and alert on overspending money
* Alternative option on cheaper money spending areas

<i><h4>Solution</i></h4>

* Model a system using m-pesa stament to predict your spending
* provide an alternative of a cheaper spending activity
* provide an alert to user when the spending of money is high(email)

In [137]:
import pandas as pd
import numpy as np
import PyPDF2
import pikepdf
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from tabula.io import read_pdf 
from dateutil import parser
# Things to do 
# install pip tabula-py for it to work
# install PyPDF2, pikepdf

In [3]:
# decrepting the file pdf
input_path = "/Users/briankimanzi/Downloads/Mpesa pdfs/Statement_All_Transactions_20240901_20250301.pdf"
output_path = "/Users/briankimanzi/Downloads/Mpesa pdfs/Tracking.pdf"
password = "102030"

with pikepdf.open(input_path, password=password) as pdf:
    pdf.save(output_path)
path = output_path

In [5]:
# getting number of pages on the pdf
with open(path, 'rb') as file:
    pdf_reader = PyPDF2.PdfReader(file)
    num_pages = len(pdf_reader.pages)

In [7]:
# extracting data from mpesa pages
def get_data():
    for page_number in range(1, num_pages+1):
        if page_number == 1:
            df =read_pdf(path, pages=page_number)
            data=df[1]
            return_df =data
        else:
            df = read_pdf(path, pages=page_number)
            data=df[0]

        if page_number != 1:
            return_df = pd.concat([return_df, data])

    return return_df.reset_index(drop=True)
working_data = get_data()

<h3><i>Data cleaning </i></h3>

In [15]:
# function to remove the commas on the dataset
def remove_comma(x):
    x = str(x)
    x = x.replace(',', '')
    return x

In [16]:
data = working_data.copy()
data['Paid in'] = data['Paid in'].apply(lambda x: remove_comma(x))
data['Withdraw\rn'] = data['Withdraw\rn'].apply(lambda x : remove_comma(x))
data['Balance'] = data['Balance'].apply(lambda x: remove_comma(x))

In [18]:
# changing data type
data = data.astype({
    'Paid in': float,
    'Withdraw\rn':float,
    'Balance':float
})

In [25]:
# dropping unwanted columns
data.drop(columns='Unnamed: 0', inplace=True)

In [126]:
# Anonymising the dataset 
def transaction(x):
    x = str(x).strip()
    if x.startswith('Merchant Payment'):
        index = x.find(' - ')+2
        name = x[index:].strip().upper()
        
        return 'BUY GOODS', name

    elif x.startswith('Deposit of funds'):
        index = x.find(' - ')
        name = x[index:].strip().upper()
        
        return 'AGENT DEPOSIT', name

    elif x.startswith('OD Loan Repayment'):
        return 'FULIZA REPAYMENT', 'FULIZA'

    elif x.startswith('OverDraft of Credit Party'):
        return 'FULIZA TAKEN', 'FULUZA'

    elif x.startswith('M-Shwari Deposit'):
        return 'M-SHWARI DEPOSIT FROM M-PESA', 'M-SHWARI'
        
    elif x.startswith('KCB M-PESA Deposit'):
        return 'KCB M-PESA DEPOSIT FROM M-PESA', 'KCB DEPOSIT'

    elif x.startswith('KCB M-PESA Withdraw'):
        return 'KCB M-PESA WITHDRAW FROM M-PESA', 'KCB WITHDRAW'

    elif x.startswith('Customer Transfer'):
        index = x.find(' - ')
        to_search = x[index:]
        last = 1

        for i in range(len(to_search)):
            try:
                int(to_search[i])
                last = i
                
            except:
                v = 10

        name = x[index+last+2:].strip().upper()
        return 'SEND MONEY', name

    elif x.startswith('M-Shwari Withdraw'):
        
        return 'M-SHWARI WITHDRAW FROM M-PESA', 'M-SHWARI WITHDRAW'

    elif x.startswith('Pay Bill'):
        if x.strip() == 'Pay Bill Charge':
            
            return 'PAY BILL CHARGES', 'TRANSACTION COST'
        else:
            index = x.find(' - ')+2
            end = x.lower().find('acc')
            name = x[index:end].strip().upper()
            
            return 'PAY BILL', name

    elif x.startswith('Funds received'):
        index = x.find(' - ')
        to_search = x[index:]
        last = 1

        for i in range(len(to_search)):
            try:
                int(to_search[i])
                last = i
            except:
                v = 10

        name = x[index+last+2:].strip().upper()
        
        return 'RECEIVED FUNDS', name

    elif x.startswith('Customer Payment to Small Business') or x.startswith('Customer Send Money'):
        index = x.find(' - ')
        to_search = x[index:]
        last = 1

        for i in range(len(to_search)):
            try:
                int(to_search[i])
                last = i
            except:
                v = 10

        name = x[index+last+2:].strip().upper()
        
        return 'POCHI LA BIASHARA', name

    elif x.startswith('Airtime Purchase'):
        return 'AIRTIME PURCHASE', 'AIRTIME'

    elif x.startswith('Business Payment From'):
        index = x.find(' - ')
        end = x.lower().find('via')
        to_search = x[index:]
        last = 1
        name = x[index:end].strip().upper()
        return 'FUNDS RECEIVED FROM BUSINESS', name

    elif x.startswith('Customer Transfer of Funds Charge'):
        return 'TRANSACTION COST', 'TRANSACTION COST'
    
    elif x.startswith('Buy Bundles Online'):
        return 'BUNDLES PURCHASE', 'BUNDES PURCHASE'
    
    elif x.startswith('Customer Withdrawal'):
        index = x.find(' - ')
        name = x[index:].strip().upper()
        return 'CASH WITHDRAWAL', name
    
    elif x.startswith('Withdrawal Charge'):
        return 'CASH WITHDRAWAL CHARGES', "TRANSACTION COST"
    
    elif x.startswith('Savings Contribution'):
        return 'TO HUSTLER FUND SAVINGS', 'HUSTLER FUND'
    
    elif x.startswith('Term Loan Disbursement for H- Fund') or x.startswith('Term Loan Disbursement for H-Fund'):
        return 'HUSTLER FUND Disbursement'.upper(), 'HUSTLER FUND'
    
    elif x.startswith('Term Loan Repayment for H- Fund') or x.startswith('Term Loan Repayment for H-Fund'):
        return 'HUSTLER FUND REPAYMENT', 'HUSTLER FUND'
    
    else:
        return 'UNIDENTIFIED', 'UNIDENTIFIED'

In [71]:
data['Details'] = data['Details'].apply(lambda x: x.replace('\r', ' '))

In [164]:
details = list(data['Details'].apply(lambda x: transaction(x)).values)
transactionType = [i[0] for i in details]
TransactionParty = [i[1] for i in details]

In [139]:
# changing the date format
def change_date(date):
    date = parser.parse(date)
    return (date.year, date.month, date.day, date.weekday(),date.hour, date.minute, date.second)

In [163]:
date = data['Completion Time'].apply(lambda x: change_date(x)).values
Year = [i[0] for i in date]
Month = [i[1] for i in date]
Date = [i[2] for i in date]
Weekday = [i[3] for i in date]
Hour = [i[4] for i in date]
Minute = [i[5] for i in date]
Seconds = [i[6] for i in date]

transactionDay = []

reverseDay = date.copy()[::-1]
datey = reverseDay[0]

d = 1

for i in reverseDay:
    if i == datey:
        transactionDay.append(d)
    else:
        datey = i
        d+= 1
        transactionDay.append(d)

transactionDay.reverse()

In [145]:
receipt = list(data['Receipt No'].values)

In [149]:
paid_in = data['Paid in'].fillna('NAN_VALUE').values

In [152]:
def withdrawAmount(x):
    try:
        if x < 0:
            return -x
        else:
            return x
    except:
        return x

In [153]:
withdraw = list(data['Withdraw\rn'].apply(lambda x: withdrawAmount(x)).values)
withdraw                

[80.0,
 1000.0,
 0.0,
 10.0,
 10.0,
 15.0,
 10.0,
 0.0,
 55.0,
 50.0,
 13.0,
 1000.0,
 100.0,
 10.0,
 15.0,
 100.0,
 55.0,
 60.0,
 20.0,
 70.0,
 50.0,
 20.0,
 15.0,
 20.0,
 120.0,
 10.0,
 0.0,
 10.0,
 10.0,
 240.0,
 10.0,
 0.0,
 35.0,
 10.0,
 20.0,
 10.0,
 10.0,
 50.0,
 7.0,
 135.0,
 20.0,
 0.0,
 30.0,
 0.0,
 10.0,
 0.0,
 50.0,
 0.0,
 25.0,
 0.0,
 40.0,
 0.0,
 30.0,
 0.0,
 10.0,
 0.0,
 2000.0,
 0.0,
 20.0,
 20.0,
 50.0,
 100.0,
 10.0,
 0.0,
 50.0,
 0.0,
 50.0,
 0.0,
 50.0,
 0.0,
 50.0,
 10.0,
 75.0,
 0.0,
 10.0,
 2.0,
 5.0,
 0.0,
 2.0,
 40.0,
 80.0,
 100.0,
 0.0,
 20.0,
 0.0,
 100.0,
 100.0,
 0.0,
 30.0,
 145.0,
 0.0,
 250.0,
 0.0,
 0.0,
 60.0,
 80.0,
 10.0,
 7.0,
 130.0,
 0.0,
 100.0,
 7.0,
 300.0,
 0.0,
 3.0,
 50.0,
 0.0,
 70.0,
 10.0,
 0.0,
 10.0,
 200.0,
 0.0,
 50.0,
 7.0,
 200.0,
 0.0,
 4800.0,
 0.0,
 10.0,
 70.0,
 60.0,
 0.0,
 100.0,
 15.0,
 10.0,
 20.0,
 10.0,
 10.0,
 7.0,
 150.0,
 20.0,
 0.0,
 15.0,
 15.0,
 15.0,
 20.0,
 10.0,
 10.0,
 10.0,
 15.0,
 15.0,
 20.0,
 50.0,
 10.0,
 2

In [155]:
transaction_Data = []

for i in range(len(paid_in)):
    try:
        x = float(paid_in[i])
        transaction_Data.append((x, 'PAID IN'))
    except:
        x = float(withdraw[i])
        transaction_Data.append((x, 'WITHDRAW'))

In [157]:
transaction_amount = [i[0] for i in transaction_Data]
paid_in_or_withdraw = [i[1] for i in transaction_Data]

In [161]:
balance =list(data["Balance"].values)

In [162]:
final_data = pd.DataFrame({
    "Receipt":receipt,
    "transaction_Day": transactionDay,
    "Year" : Year,
    "Month" : Month,
    "Date" : Date,
    "Weekday" : Weekday,
    "Hour" : Hour,
    "Minute" : Minute,
    "Seconds" : Seconds,
    "Transaction_type" : transactionType,
    "Transaction_party" : TransactionParty,
    "Transaction_amount" : transaction_amount,
    "paid_in_or_Withdraw" : paid_in_or_withdraw,
    "Balance" : balance
})

[27.0,
 107.0,
 1107.0,
 7.0,
 17.0,
 27.0,
 42.0,
 52.0,
 2.0,
 57.0,
 107.0,
 120.0,
 1120.0,
 1220.0,
 1230.0,
 1245.0,
 1345.0,
 1400.0,
 1460.0,
 1480.0,
 1550.0,
 1600.0,
 1620.0,
 1635.0,
 1655.0,
 1775.0,
 1785.0,
 1735.0,
 1745.0,
 1755.0,
 1995.0,
 2005.0,
 5.0,
 40.0,
 50.0,
 70.0,
 80.0,
 90.0,
 140.0,
 147.0,
 282.0,
 302.0,
 2.0,
 32.0,
 2.0,
 12.0,
 1.0,
 51.0,
 11.0,
 36.0,
 11.0,
 51.0,
 1.0,
 31.0,
 1.0,
 11.0,
 1.0,
 2001.0,
 1.0,
 21.0,
 41.0,
 91.0,
 191.0,
 201.0,
 1.0,
 51.0,
 1.0,
 51.0,
 1.0,
 51.0,
 1.0,
 51.0,
 61.0,
 136.0,
 36.0,
 46.0,
 48.0,
 53.0,
 3.0,
 5.0,
 45.0,
 125.0,
 225.0,
 25.0,
 45.0,
 25.0,
 125.0,
 225.0,
 25.0,
 55.0,
 200.0,
 0.0,
 250.0,
 220.0,
 30.0,
 90.0,
 170.0,
 180.0,
 187.0,
 317.0,
 117.0,
 217.0,
 224.0,
 524.0,
 24.0,
 27.0,
 77.0,
 27.0,
 97.0,
 107.0,
 7.0,
 17.0,
 217.0,
 17.0,
 67.0,
 74.0,
 274.0,
 74.0,
 4874.0,
 24.0,
 34.0,
 104.0,
 164.0,
 114.0,
 214.0,
 229.0,
 239.0,
 259.0,
 269.0,
 279.0,
 286.0,
 436.0,
 456.0,
 